# 🌿 Détection et Classification Automatique de Maladies des Plantes
## Projet Académique — Machine Learning & Deep Learning

**Pipeline complet :** Dataset → Prétraitement → Segmentation → Extraction Features → Classification ML/DL → Évaluation → Benchmark

**12 classes cibles :** Maïs (×4) | Pomme de terre (×3) | Tomate (×5)

---

## 1. Imports & Configuration


In [ ]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2
import joblib

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score,
                             precision_score, recall_score)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Chemins
RAW_DIR       = Path('./data/raw')
PROCESSED_DIR = Path('./data/processed')
MODEL_DIR     = Path('./models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CLASSES_12 = sorted([
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight',
    'Corn_(maize)___healthy',
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___healthy',
])
print(f'TensorFlow : {tf.__version__}')
print(f'Classes    : {len(CLASSES_12)}')


## 2. Dataset

**Source :** PlantVillage Dataset (Kaggle)

**12 classes sélectionnées :**
- Maïs × 4 (saine, cercospora, rouille, brûlure nordique)
- Pomme de terre × 3 (saine, alternariose, mildiou)
- Tomate × 5 (saine, alternariose, mildiou, cladosporiose, septoriose)


In [ ]:
# Analyse de la distribution des classes
class_counts = {}
for d in sorted(RAW_DIR.iterdir()):
    if d.is_dir():
        imgs = list(d.glob('*.*'))
        class_counts[d.name] = len(imgs)

df_dist = pd.DataFrame(list(class_counts.items()),
                        columns=['Classe', 'Nombre d images'])
print(df_dist.to_string(index=False))
print(f"\nTotal : {df_dist['Nombre d images'].sum()} images")

fig, ax = plt.subplots(figsize=(14, 5))
ax.barh(df_dist['Classe'], df_dist['Nombre d images'], color='steelblue')
ax.set_xlabel('Nombre d images')
ax.set_title('Distribution des images par classe')
plt.tight_layout()
plt.show()


## 3. Prétraitement des Images

**Split stratifié 70/15/15** avec graine aléatoire fixée (seed=42) pour assurer la **reproductibilité**.

Chaque sous-ensemble contient exactement les mêmes 12 classes.


In [ ]:
# Vérification du split
for split in ['train', 'val', 'test']:
    split_dir = PROCESSED_DIR / split
    if split_dir.exists():
        total = sum(len(list((split_dir/c).glob('*.*')))
                    for c in os.listdir(split_dir)
                    if (split_dir/c).is_dir())
        n_cls = len([d for d in split_dir.iterdir() if d.is_dir()])
        print(f'  {split.upper():<6} : {n_cls:>2} classes | {total:>6} images')
    else:
        print(f'  {split.upper():<6} : NON GÉNÉRÉ — lancez splitter.py')


In [ ]:
# Exemples d images par classe
VALID_EXT = {'.jpg','.jpeg','.png','.bmp'}
sample_classes = list(CLASSES_12[:6])
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, cls in zip(axes.flatten(), sample_classes):
    cls_dir = PROCESSED_DIR / 'train' / cls
    if cls_dir.exists():
        imgs = [f for f in cls_dir.iterdir() if f.suffix.lower() in VALID_EXT]
        if imgs:
            img = cv2.imread(str(imgs[0]))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224,224))
            ax.imshow(img)
    short_name = cls.split('___')[-1].replace('_',' ')
    ax.set_title(short_name, fontsize=9)
    ax.axis('off')
plt.suptitle('Exemples d images du dataset (224×224)', fontsize=13)
plt.tight_layout()
plt.show()


## 4. Segmentation des Feuilles (Vision Classique)

**Algorithme :** Double masque HSV
- Masque 1 — Vert / Jaune-vert (H: 22–95) : feuilles saines / légèrement malades
- Masque 2 — Brun / Orange (H: 5–22) : zones nécrosées avancées

Opérations morphologiques (OPEN + CLOSE) pour supprimer le bruit.


In [ ]:
from src.vision.segmentation import segment_leaf

# Démonstration sur 3 images
demo_imgs = []
for cls in ['Tomato___Late_blight', 'Corn_(maize)___Common_rust_', 'Potato___healthy']:
    cls_dir = PROCESSED_DIR / 'train' / cls
    if cls_dir.exists():
        files = [f for f in cls_dir.iterdir() if f.suffix.lower() in VALID_EXT]
        if files:
            demo_imgs.append((cls, str(files[0])))

fig, axes = plt.subplots(len(demo_imgs), 3, figsize=(13, 4*len(demo_imgs)))
if len(demo_imgs) == 1: axes = [axes]
for row, (cls, path) in enumerate(demo_imgs):
    orig, mask, seg = segment_leaf(path)
    titles = ['Original', 'Masque HSV (binaire)', 'Feuille segmentée']
    imgs_row = [orig, mask, seg]
    for col, (img_show, ttl) in enumerate(zip(imgs_row, titles)):
        axes[row][col].imshow(img_show, cmap='gray' if col==1 else None)
        axes[row][col].set_title(f"{ttl}\n{cls.split('___')[-1]}", fontsize=9)
        axes[row][col].axis('off')
plt.suptitle('Segmentation par double masque HSV', fontsize=13)
plt.tight_layout()
plt.show()


## 5. Extraction de Caractéristiques

| Catégorie | Méthode | Dimensions |
|-----------|---------|------------|
| **Couleur** | Histogramme HSV 3D (8×8×8 bins) | 512 |
| **Texture** | GLCM Haralick (contraste, énergie, corrélation…) | 20 |
| **Forme**   | Moments de Hu (×7) + Circularité + Compacité + Ratio aire | 10 |
| **Total** | | **542** |


In [ ]:
from src.features.extractors import (
    extract_color_histogram, extract_glcm_texture,
    extract_shape_features, extract_features
)

# Test sur une image
test_cls = CLASSES_12[0]
test_dir = PROCESSED_DIR / 'train' / test_cls
test_img_path = next(f for f in test_dir.iterdir()
                     if f.suffix.lower() in VALID_EXT)

img_rgb, mask, _ = segment_leaf(str(test_img_path))
color_f   = extract_color_histogram(img_rgb, mask)
texture_f = extract_glcm_texture(img_rgb, mask)
shape_f   = extract_shape_features(mask)
all_f     = extract_features(img_rgb, mask)

print(f'Couleur  (HSV hist) : {color_f.shape}  — {color_f[:5].round(3)}')
print(f'Texture  (GLCM)     : {texture_f.shape}  — {texture_f.round(3)}')
print(f'Forme    (Hu+métr.) : {shape_f.shape}  — {shape_f.round(4)}')
print(f'Vecteur complet     : {all_f.shape}')


## 6. Classification — Random Forest

- `n_estimators=300`, `class_weight='balanced'`
- Features normalisées par `StandardScaler`
- Entraîné sur **tout** le set d'entraînement (sans limite artificielle)


In [ ]:
# Chargement du modèle RF déjà entraîné
rf_path = MODEL_DIR / 'rf_model.pkl'
if rf_path.exists():
    rf_data = joblib.load(rf_path)
    rf_model = rf_data['model']
    rf_scaler = rf_data['scaler']
    classes = rf_data['classes']
    print(f'RF chargé — {len(classes)} classes')
    print(f'Paramètres : {rf_model.get_params()}')
else:
    print('Modèle RF non trouvé. Lancez : python train_ml.py')


In [ ]:
# Matrice de confusion RF
from PIL import Image
if Path('confusion_matrix_rf.png').exists():
    img = Image.open('confusion_matrix_rf.png')
    plt.figure(figsize=(14,11))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Matrice de Confusion — Random Forest')
    plt.tight_layout()
    plt.show()
else:
    print('Lancez train_ml.py pour générer la matrice de confusion.')


## 7. Classification — SVM (Support Vector Machine)

- Kernel **RBF** avec `C=10`, `gamma='scale'`
- `class_weight='balanced'` pour compenser le déséquilibre
- `probability=True` pour les prédictions Top-3


In [ ]:
svm_path = MODEL_DIR / 'svm_model.pkl'
if svm_path.exists():
    svm_data = joblib.load(svm_path)
    svm_model = svm_data['model']
    print(f'SVM chargé — {len(svm_data["classes"])} classes')
    print(f'Paramètres : {svm_model.get_params()}')
else:
    print('Modèle SVM non trouvé. Lancez : python train_ml.py')


In [ ]:
if Path('confusion_matrix_svm.png').exists():
    img = Image.open('confusion_matrix_svm.png')
    plt.figure(figsize=(14,11))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Matrice de Confusion — SVM (RBF)')
    plt.tight_layout()
    plt.show()


## 8. Deep Learning — MobileNetV2 (Transfer Learning)

**Architecture :**
```
Input (224×224×3)
   → Data Augmentation (Flip, Rotation, Zoom)
   → preprocess_input [-1, 1]
   → MobileNetV2 (ImageNet weights) — couches gelées en Phase 1
   → GlobalAveragePooling2D
   → BatchNormalization
   → Dropout(0.3)
   → Dense(12, softmax)
```
**Phase 1** : Feature Extraction (base gelée, lr=1e-3)

**Phase 2** : Fine-tuning (30 dernières couches dégelées, lr=1e-5)


In [ ]:
dl_path = MODEL_DIR / 'mobilenetv2_plants.keras'
if dl_path.exists():
    dl_model = tf.keras.models.load_model(str(dl_path), compile=False)
    dl_model.summary()
else:
    print('Modèle DL non trouvé. Lancez : python train_dl.py')


In [ ]:
# Courbes d apprentissage
if Path('training_history.png').exists():
    img = Image.open('training_history.png')
    plt.figure(figsize=(14,5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Courbes Accuracy & Loss — Phase 1 + Fine-tuning')
    plt.show()


In [ ]:
# Matrice de confusion DL
if Path('confusion_matrix_dl.png').exists():
    img = Image.open('confusion_matrix_dl.png')
    plt.figure(figsize=(14,11))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Matrice de Confusion — MobileNetV2')
    plt.show()


## 9. Explicabilité — Grad-CAM

**Gradient-weighted Class Activation Mapping** permet de visualiser quelles zones de l'image ont influencé la décision du réseau.

- Couche cible : `out_relu` (dernière activation spatiale de MobileNetV2, 7×7×1280)
- Les gradients de la classe prédite sont moyennés spatialement
- La combinaison pondérée produit une carte de chaleur (rouge = zone décisive)


In [ ]:
from predict import make_gradcam_heatmap, blend_gradcam

dl_path = MODEL_DIR / 'mobilenetv2_plants.keras'
if not dl_path.exists():
    print('DL model required.')
else:
    dl_model = tf.keras.models.load_model(str(dl_path), compile=False)
    demo_classes = ['Tomato___Late_blight', 'Corn_(maize)___Common_rust_',
                    'Potato___Early_blight']
    demo_paths = []
    for cls in demo_classes:
        d = PROCESSED_DIR / 'test' / cls
        if d.exists():
            files = [f for f in d.iterdir() if f.suffix.lower() in VALID_EXT]
            if files: demo_paths.append((cls, str(files[0])))

    fig, axes = plt.subplots(len(demo_paths), 3, figsize=(13, 4.5*len(demo_paths)))
    if len(demo_paths)==1: axes=[axes]
    for row,(cls,path) in enumerate(demo_paths):
        img = tf.keras.utils.load_img(path, target_size=(224,224))
        arr = tf.keras.utils.img_to_array(img)
        batch = tf.expand_dims(arr, 0)
        heatmap = make_gradcam_heatmap(batch, dl_model)
        blended = blend_gradcam(path, heatmap)
        orig_rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        hm_up = cv2.resize(heatmap, (224,224))
        cols_data = [cv2.resize(orig_rgb,(224,224)), hm_up, blended]
        ttls = ['Original', 'Heatmap Grad-CAM', 'Superposition']
        for col,(d,t) in enumerate(zip(cols_data,ttls)):
            axes[row][col].imshow(d, cmap='hot' if col==1 else None)
            axes[row][col].set_title(f'{t}\n{cls.split("___")[-1]}', fontsize=9)
            axes[row][col].axis('off')
    plt.suptitle('Grad-CAM — Explicabilité du modèle Deep Learning', fontsize=13)
    plt.tight_layout()
    plt.show()


## 10. Benchmark ML vs Deep Learning

Comparaison des trois modèles sur le **même test set** avec 5 métriques :
Accuracy, Précision, Rappel, F1-score (weighted), Temps d'inférence


In [ ]:
if Path('benchmark_results.csv').exists():
    df_bench = pd.read_csv('benchmark_results.csv')
    print(df_bench.to_string(index=False))
else:
    print('Lancez : python src/utils/benchmark.py')


In [ ]:
if Path('benchmark_chart.png').exists():
    from PIL import Image
    img = Image.open('benchmark_chart.png')
    plt.figure(figsize=(13,6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Comparaison Benchmark — RF vs SVM vs MobileNetV2')
    plt.show()


## 11. Conclusion & Recommandations

### Synthèse des résultats

| Critère | Random Forest | SVM (RBF) | MobileNetV2 |
|---------|--------------|-----------|-------------|
| **Accuracy** | ~77% | ~80% | ~92% |
| **Interprétabilité** | Importance features | Kernel SHAP | Grad-CAM |
| **Vitesse inférence** | Très rapide | Rapide | Modérée (CPU) |
| **Mémoire modèle** | ~7 Mo | ~15 Mo | ~10 Mo |

### Points forts du Deep Learning
- Précision nettement supérieure grâce au Transfer Learning (ImageNet)
- Robuste aux variations de luminosité, angle et fond
- Grad-CAM offre une explicabilité visuelle forte pour les agriculteurs

### Points forts du Machine Learning classique
- Entraînable sans GPU, rapide sur CPU
- Features interprétables (GLCM, Hu Moments)
- Idéal pour déploiement sur systèmes embarqués

### Perspectives d'amélioration
1. Augmenter le dataset (images de terrain en conditions réelles)
2. Tester EfficientNetV2 ou ConvNeXt pour encore plus d'accuracy
3. Déploiement mobile via TFLite (quantification int8)
4. Intégration d'une API REST pour usage en télédétection agricole

---
*Projet réalisé avec PlantVillage Dataset | TensorFlow 2.x | scikit-learn*
